# 3.35 — Cost-Sensitive Learning

Cost-sensitive learning changes the question from "how many mistakes did we make?" to "how expensive were those mistakes?" In this lesson, we build the idea from empirical risk, a class-cost matrix, threshold decisions, model-selection costs, and stabilization, using only tiny NumPy arrays whose arithmetic can be checked by hand.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build cost-sensitive learning one idea at a time. Run each cell in order and read the printed intermediate values — every cost, average, threshold, and comparison is visible. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized costs, and tiny optimization loops.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any randomized toy data.

### 1. A cost matrix turns mistakes into dollars, minutes, or risk

Ordinary accuracy treats every wrong label as one mistake. Cost-sensitive learning starts by writing down a **cost matrix** `C_w`, where row `a` is the true class and column `b` is the predicted class. The diagonal is usually zero because a correct decision has no error cost. The off-diagonal entries need not match: missing a fraud case may cost much more than investigating a safe case.

In [ ]:
labels_w = np.array(["safe", "risky"])  # class 0 = safe, class 1 = risky.
C_w = np.array([[0.0, 1.0],   # true safe: predicting risky costs one unnecessary review.
                [8.0, 0.0]])  # true risky: predicting safe misses a costly risky case.
print("cost matrix rows=true, cols=predicted:\n", C_w)
print("false alarm cost:", C_w[0, 1], "missed risky cost:", C_w[1, 0])
assert C_w[1, 0] == 8.0 and C_w[0, 1] == 1.0

▶ What you'll see: the missed-risky error is eight times as expensive as the false-alarm error.

In [ ]:
plt.figure(figsize=(4.2, 3.2))
plt.imshow(C_w, cmap="Reds")
plt.colorbar(label="cost")
plt.xticks([0, 1], labels_w); plt.yticks([0, 1], labels_w)
plt.xlabel("predicted class"); plt.ylabel("true class")
plt.title("1: asymmetric decision costs")
plt.show()

▶ What you'll see: the bright lower-left cell marks the high-cost mistake: true risky predicted safe.

*Why it's done this way:* the learner can only optimize what the loss function says matters. A matrix makes the contract explicit: `C[a,b]` is the penalty for choosing action `b` in state `a`, so two classifiers with the same error count can have very different risk when their mistakes fall in different cells.

### 2. Empirical cost is an average over examples

The mathematical object is expected cost, $R(f)=\mathbb E[C_{Y,f(X)}]$. In a notebook we approximate that expectation with an empirical average: look up the cost of each observed `(true, predicted)` pair and average. This is the same empirical-risk move used throughout ML, but the per-example loss is now a cost lookup instead of a 0/1 indicator.

In [ ]:
y_true_w = np.array([0, 1, 1, 0, 1, 0])  # six validation labels.
y_pred_w = np.array([0, 0, 1, 1, 0, 0])  # predictions from one rule.
per_example_cost_w = C_w[y_true_w, y_pred_w]  # vectorized lookup: C[true_i, pred_i].
print("per-example costs:", per_example_cost_w)
print("raw mistakes:", int(np.sum(y_true_w != y_pred_w)))

▶ What you'll see: three mistakes, but their costs are `8, 1, 8`, not all `1`.

In [ ]:
empirical_cost_w = float(np.mean(per_example_cost_w))
print("empirical cost:", round(empirical_cost_w, 3))
assert round(empirical_cost_w, 3) == 2.833

plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(per_example_cost_w)), per_example_cost_w, color="indianred")
plt.axhline(empirical_cost_w, color="black", linestyle="--", label="mean cost")
plt.xlabel("example"); plt.ylabel("cost")
plt.title("2: empirical cost is an average")
plt.legend(); plt.show()

▶ What you'll see: two high bars dominate the average even though only three examples are wrong.

*Why it's done this way:* expectation is the long-run average cost of deploying the rule. The sample average is its finite-data estimate, and vectorized indexing mirrors the formula exactly: each example contributes one `C[true,pred]`, then the learner minimizes their mean.

### 3. A probability threshold should minimize expected action cost

When a model outputs $p=P(Y=1\mid x)$, we should not automatically predict class 1 at `0.5`. For two classes with `C[0,1]=c_fp` and `C[1,0]=c_fn`, predicting class 1 has expected cost `(1-p)c_fp`, while predicting class 0 has expected cost `p c_fn`. Choose class 1 when `(1-p)c_fp < p c_fn`, giving the threshold $p>c_{fp}/(c_{fp}+c_{fn})$.

In [ ]:
c_fp_w = C_w[0, 1]
c_fn_w = C_w[1, 0]
threshold_w = c_fp_w / (c_fp_w + c_fn_w)
print("cost-sensitive threshold:", round(threshold_w, 3))
assert round(threshold_w, 3) == 0.111

▶ What you'll see: because false negatives are expensive, we predict risky once probability exceeds only about 0.111.

In [ ]:
p_grid_w = np.linspace(0, 1, 101)
cost_predict_safe_w = p_grid_w * c_fn_w
cost_predict_risky_w = (1 - p_grid_w) * c_fp_w
plt.figure(figsize=(4.8, 3.2))
plt.plot(p_grid_w, cost_predict_safe_w, label="predict safe")
plt.plot(p_grid_w, cost_predict_risky_w, label="predict risky")
plt.axvline(threshold_w, color="black", linestyle="--", label="threshold")
plt.xlabel("p(risky | x)"); plt.ylabel("expected cost")
plt.title("3: choose the cheaper action")
plt.legend(); plt.show()

▶ What you'll see: the two cost lines cross near 0.111; to the right, predicting risky is cheaper.

*Why it's done this way:* the threshold is not a tuning trick but a Bayes decision rule. We compare the cost of each possible action under the model's posterior uncertainty, so the optimal decision boundary moves toward catching costly positives when misses are expensive.

### 4. Reweighting a loss is a training-time approximation to the same idea

Some algorithms cannot consume a full cost matrix directly. A common workaround is to weight each training example by the cost of getting its class wrong. Positive examples receive a larger weight when false negatives are expensive, so the optimizer feels their errors more strongly.

In [ ]:
losses_w = np.array([0.268, 0.109, 0.420])  # verified toy losses from the lesson text.
raw_risk_w = float(np.mean(losses_w))
print("losses:", losses_w)
print("raw empirical risk:", round(raw_risk_w, 3))
assert round(raw_risk_w, 3) == 0.266

▶ What you'll see: `(0.268 + 0.109 + 0.420) / 3 = 0.266`, the raw fit term.

In [ ]:
weights_w = np.array([1.0, 3.0, 8.0])  # costlier examples carry more consequence.
weighted_risk_w = float(np.sum(weights_w * losses_w) / np.sum(weights_w))
print("weighted empirical risk:", round(weighted_risk_w, 3))
assert round(weighted_risk_w, 3) == 0.330

plt.figure(figsize=(4.5, 3))
plt.bar(["unweighted", "weighted"], [raw_risk_w, weighted_risk_w], color=["gray", "crimson"])
plt.ylabel("average loss")
plt.title("4: costly examples pull harder")
plt.show()

▶ What you'll see: weighting changes the training objective even with the same three loss numbers.

*Why it's done this way:* multiplying losses by weights changes the empirical distribution seen by the optimizer. It approximates minimizing expected cost by making an error on an expensive class count like many ordinary errors, which bends the fitted model toward protecting that class.

### 5. Model selection uses the full decision score, not the prettiest raw fit

The lesson text emphasizes that the raw training number is only one term. If a method carries complexity, regularization, operational, or review cost, the score used for selection is `raw risk + method cost`. A more flexible model can look attractive on raw fit and still lose after the cost is included.

In [ ]:
method_cost_w = 0.100
score_w = raw_risk_w + method_cost_w
alternative_score_w = 0.402
gap_w = alternative_score_w - score_w
relative_gap_w = gap_w / alternative_score_w
print("score = raw risk + cost:", round(score_w, 3))
print("gap vs flexible alternative:", round(gap_w, 3))
print("relative gap:", round(relative_gap_w, 3))
assert round(score_w, 3) == 0.366
assert round(gap_w, 3) == 0.036
assert round(relative_gap_w, 3) == 0.090

▶ What you'll see: the lower complete score is 0.366, beating the flexible alternative by 0.036.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["raw", "+ method cost", "alternative"], [raw_risk_w, score_w, alternative_score_w],
        color=["lightgray", "seagreen", "darkorange"])
plt.ylabel("decision score")
plt.title("5: compare complete scores")
plt.show()

▶ What you'll see: the raw bar is not the selection unit; the green complete-score bar is.

*Why it's done this way:* empirical risk alone rewards fit, while deployment decisions care about fit plus the price of achieving it. Adding the method cost keeps the comparison on one scale, so a tiny training gain must earn back the extra complexity it introduced.

### 6. Stabilization can win by reducing future cost

A stabilizing knob might be stronger regularization, a simpler threshold policy, or a conservative operating constraint. In the lesson's verified toy case, stabilization reduces the decision score by 20%, so the final comparison is among baseline, flexible, and stabilized scores.

In [ ]:
stable_score_w = 0.80 * score_w
scores_w = np.array([score_w, alternative_score_w, stable_score_w])
names_w = np.array(["baseline", "flexible", "stabilized"])
winner_w = int(np.argmin(scores_w))
print("stable score:", round(stable_score_w, 3))
print("winner:", names_w[winner_w], round(scores_w[winner_w], 3))
assert round(stable_score_w, 3) == 0.293
assert names_w[winner_w] == "stabilized"

▶ What you'll see: `0.293` is the minimum of the three complete decision scores.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(names_w, scores_w, color=["seagreen", "darkorange", "royalblue"])
plt.ylabel("complete decision score")
plt.title("6: lower expected cost wins")
plt.show()

▶ What you'll see: the stabilized bar is lowest, so it is the model carried forward.

*Why it's done this way:* stabilization deliberately gives up brittle variation when that variation is likely to create future cost. The winner is not the fanciest model; it is the rule with the lowest full score on the scale defined by the decision problem.

## 🛠️ Setup

In [ ]:
import numpy as np  # NumPy supplies arrays, vectorized cost lookups, and tiny optimization loops.
import matplotlib.pyplot as plt  # Matplotlib supplies the small diagnostic plots in the examples.
np.random.seed(0)  # Fix the global seed so any stochastic example is reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Build a cost matrix

**Goal.** Encode asymmetric costs for a binary decision, because cost-sensitive learning starts with the consequences of each action. We build it in 2 steps.

In [ ]:
labels_b1 = np.array(["safe", "risky"])  # Name the two classes for readable output.
C_b1 = np.array([[0.0, 1.0], [8.0, 0.0]])  # Rows are true labels; columns are predicted labels.
print("labels:", labels_b1)  # Inspect class order before reading the matrix.
print("C_b1:\n", C_b1)  # Inspect the cost table.
assert C_b1[1, 0] == 8.0  # Verify the costly false negative.

▶ What you'll see: a 2×2 matrix where one off-diagonal mistake is much brighter in meaning than the other.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact heatmap.
plt.imshow(C_b1, cmap="Reds")  # Draw costs as colors.
plt.colorbar(label="cost")  # Add a numeric scale.
plt.xticks([0, 1], labels_b1); plt.yticks([0, 1], labels_b1)  # Label predicted and true classes.
plt.title("Basic 1: cost matrix")  # Title the diagnostic plot.
plt.xlabel("predicted"); plt.ylabel("true")  # Clarify the matrix orientation.
plt.show()  # Display the plot.

▶ What you'll see: the high-cost cell is true risky but predicted safe.

👀 Takeaway: a cost matrix defines the scale the learner should optimize.

### Basic 2 — Look up per-example costs

**Goal.** Convert true and predicted labels into one cost per example, because empirical risk is built from those losses. We build it in 2 steps.

In [ ]:
C_b2 = np.array([[0.0, 1.0], [8.0, 0.0]])  # Recreate the cost matrix locally.
y_true_b2 = np.array([0, 1, 1, 0])  # Define four true labels.
y_pred_b2 = np.array([0, 0, 1, 1])  # Define four predictions.
print("true:", y_true_b2, "pred:", y_pred_b2)  # Inspect paired labels.

▶ What you'll see: examples 1 and 3 are mistakes, but they are not equally expensive.

In [ ]:
costs_b2 = C_b2[y_true_b2, y_pred_b2]  # Look up C[true_i, pred_i] for each example.
print("costs:", costs_b2)  # Inspect the per-example costs.
assert np.allclose(costs_b2, [0.0, 8.0, 0.0, 1.0])  # Verify the cost lookup.
plt.figure(figsize=(4, 3))  # Create a compact bar chart.
plt.bar(np.arange(len(costs_b2)), costs_b2, color="indianred")  # Show cost per example.
plt.title("Basic 2: per-example costs")  # Title the plot.
plt.xlabel("example"); plt.ylabel("cost")  # Label axes.
plt.show()  # Display the plot.

▶ What you'll see: one false negative contributes eight times the false-alarm bar.

👀 Takeaway: cost-sensitive loss is a vector of consequences, not just a vector of wrong/right flags.

### Basic 3 — Average empirical cost

**Goal.** Compute the sample-average cost, because empirical risk estimates future expected cost from observed examples. We build it in 2 steps.

In [ ]:
costs_b3 = np.array([0.0, 8.0, 0.0, 1.0])  # Reuse the four example costs from Basic 2.
total_b3 = float(np.sum(costs_b3))  # Sum the observed costs.
print("total cost:", total_b3)  # Inspect the numerator of the average.

▶ What you'll see: the total is dominated by the single cost-8 mistake.

In [ ]:
risk_b3 = float(np.mean(costs_b3))  # Divide by the number of examples.
print("empirical risk:", round(risk_b3, 3))  # Inspect the average cost.
assert round(risk_b3, 3) == 2.250  # Verify 9 / 4 = 2.25.
plt.figure(figsize=(4, 3))  # Create a compact plot.
plt.bar(["total", "average"], [total_b3, risk_b3], color=["gray", "seagreen"])  # Compare sum and mean.
plt.title("Basic 3: sum then average")  # Title the plot.
plt.ylabel("cost")  # Label the cost scale.
plt.show()  # Display the plot.

▶ What you'll see: the average rescales total cost back to a per-example number.

👀 Takeaway: empirical cost is the average consequence per validation example.

### Basic 4 — Compare accuracy with cost

**Goal.** Show two rules with the same number of mistakes but different total cost, because accuracy ignores which mistakes were made. We build it in 3 steps.

In [ ]:
C_b4 = np.array([[0.0, 1.0], [8.0, 0.0]])  # Cost matrix with expensive misses.
y_true_b4 = np.array([0, 1, 1, 0, 1, 0])  # Six labels.
pred_A_b4 = np.array([0, 0, 1, 1, 1, 0])  # Rule A makes one miss and one false alarm.
pred_B_b4 = np.array([1, 1, 1, 1, 0, 0])  # Rule B makes two false alarms and one miss pattern.
print("mistakes A/B:", np.sum(pred_A_b4 != y_true_b4), np.sum(pred_B_b4 != y_true_b4))  # Count errors.

▶ What you'll see: the mistake counts are close, but counts alone hide severity.

In [ ]:
cost_A_b4 = float(np.mean(C_b4[y_true_b4, pred_A_b4]))  # Average cost for rule A.
cost_B_b4 = float(np.mean(C_b4[y_true_b4, pred_B_b4]))  # Average cost for rule B.
acc_A_b4 = float(np.mean(pred_A_b4 == y_true_b4))  # Accuracy for rule A.
acc_B_b4 = float(np.mean(pred_B_b4 == y_true_b4))  # Accuracy for rule B.
print("accuracy A/B:", round(acc_A_b4, 3), round(acc_B_b4, 3))  # Inspect count-based quality.
print("cost A/B:", round(cost_A_b4, 3), round(cost_B_b4, 3))  # Inspect consequence-based quality.
assert round(cost_A_b4, 3) == 1.500 and round(cost_B_b4, 3) == 1.667

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(4.5, 3))  # Create a compact comparison plot.
plt.bar(["A cost", "B cost"], [cost_A_b4, cost_B_b4], color=["seagreen", "darkorange"])  # Compare average costs.
plt.title("Basic 4: cost can change the ranking")  # Title the plot.
plt.ylabel("average cost")  # Label the decision scale.
plt.show()  # Display the plot.

▶ What you'll see: the cost bars reveal which rule has lower consequence even when accuracy is similar.

👀 Takeaway: accuracy is the wrong objective when mistakes have unequal consequences.

### Basic 5 — Derive the cost-sensitive threshold

**Goal.** Compute the binary threshold from false-positive and false-negative costs, because the best decision boundary depends on consequences. We build it in 2 steps.

In [ ]:
c_fp_b5 = 1.0  # Cost of predicting risky when the case is safe.
c_fn_b5 = 8.0  # Cost of predicting safe when the case is risky.
threshold_b5 = c_fp_b5 / (c_fp_b5 + c_fn_b5)  # Bayes threshold where action costs tie.
print("threshold:", round(threshold_b5, 3))  # Inspect the decision cutoff.
assert round(threshold_b5, 3) == 0.111  # Verify 1 / 9.

▶ What you'll see: the threshold is far below 0.5 because misses are expensive.

In [ ]:
probs_b5 = np.array([0.05, 0.12, 0.40, 0.80])  # Example predicted probabilities for risky.
preds_b5 = (probs_b5 > threshold_b5).astype(int)  # Apply the cost-sensitive threshold.
print("probabilities:", probs_b5)  # Inspect scores.
print("predictions:", preds_b5)  # Inspect cost-aware decisions.
plt.figure(figsize=(4, 3))  # Create a threshold plot.
plt.scatter(probs_b5, np.zeros_like(probs_b5), c=preds_b5, cmap="coolwarm", s=80)  # Color decisions.
plt.axvline(threshold_b5, color="black", linestyle="--")  # Mark threshold.
plt.yticks([]); plt.xlabel("p(risky)")  # Keep the plot focused on the cutoff.
plt.title("Basic 5: low threshold for costly positives")  # Title the plot.
plt.show()  # Display the plot.

▶ What you'll see: probabilities above about 0.111 become risky predictions.

👀 Takeaway: threshold choice is a mathematical consequence of the cost ratio.

### Basic 6 — Compare action costs for one probability

**Goal.** Choose between two actions by expected cost, because the probability alone is not the decision. We build it in 2 steps.

In [ ]:
p_risky_b6 = 0.20  # Model probability that the case is risky.
c_fp_b6, c_fn_b6 = 1.0, 8.0  # False-alarm and missed-risky costs.
cost_safe_b6 = p_risky_b6 * c_fn_b6  # If we predict safe, only risky cases create missed-risky cost.
cost_risky_b6 = (1 - p_risky_b6) * c_fp_b6  # If we predict risky, only safe cases create false-alarm cost.
print("cost predict safe:", round(cost_safe_b6, 3))  # Inspect action 0 cost.
print("cost predict risky:", round(cost_risky_b6, 3))  # Inspect action 1 cost.
assert round(cost_safe_b6, 3) == 1.600 and round(cost_risky_b6, 3) == 0.800

▶ What you'll see: at probability 0.20, predicting risky is cheaper despite risk being less than 50% likely.

In [ ]:
decision_b6 = "risky" if cost_risky_b6 < cost_safe_b6 else "safe"  # Choose the lower-cost action.
print("decision:", decision_b6)  # Inspect the chosen class.
plt.figure(figsize=(4, 3))  # Create a compact action-cost chart.
plt.bar(["predict safe", "predict risky"], [cost_safe_b6, cost_risky_b6], color=["gray", "crimson"])  # Compare costs.
plt.title("Basic 6: action with lower expected cost")  # Title the plot.
plt.ylabel("expected cost")  # Label the scale.
plt.show()  # Display the plot.

▶ What you'll see: the smaller bar corresponds to the chosen action.

👀 Takeaway: the optimal class is the cheaper action under the current probability and cost matrix.

### Basic 7 — Reweight per-example losses

**Goal.** Weight training losses by class consequence, because many optimizers accept weights more easily than a full cost matrix. We build it in 2 steps.

In [ ]:
losses_b7 = np.array([0.268, 0.109, 0.420])  # Three verified toy losses from the lesson.
weights_b7 = np.array([1.0, 3.0, 8.0])  # Give costly examples more influence.
print("losses:", losses_b7)  # Inspect raw losses.
print("weights:", weights_b7)  # Inspect importance weights.

▶ What you'll see: the costlier examples will dominate the weighted average.

In [ ]:
unweighted_b7 = float(np.mean(losses_b7))  # Ordinary empirical risk.
weighted_b7 = float(np.sum(weights_b7 * losses_b7) / np.sum(weights_b7))  # Weighted empirical risk.
print("unweighted:", round(unweighted_b7, 3), "weighted:", round(weighted_b7, 3))  # Compare objectives.
assert round(unweighted_b7, 3) == 0.266 and round(weighted_b7, 3) == 0.330
plt.figure(figsize=(4, 3))  # Create comparison bars.
plt.bar(["plain", "weighted"], [unweighted_b7, weighted_b7], color=["gray", "purple"])  # Show objective shift.
plt.title("Basic 7: weighted empirical risk")  # Title the plot.
plt.ylabel("risk")  # Label y-axis.
plt.show()  # Display the plot.

▶ What you'll see: weighting changes the objective from 0.266 to about 0.330.

👀 Takeaway: weights make costly examples count more in the training objective.

### Basic 8 — Add a method cost

**Goal.** Add an operational or complexity cost to raw empirical risk, because model selection should compare complete decision scores. We build it in 2 steps.

In [ ]:
raw_risk_b8 = 0.266  # Raw empirical risk from the verified toy average.
method_cost_b8 = 0.100  # Complexity, regularization, or operational cost.
score_b8 = raw_risk_b8 + method_cost_b8  # Complete selection score.
print("raw risk:", raw_risk_b8)  # Inspect the fit term.
print("method cost:", method_cost_b8)  # Inspect the added cost term.

▶ What you'll see: the method cost is separate from the raw fit.

In [ ]:
print("complete score:", round(score_b8, 3))  # Inspect the score used for selection.
assert round(score_b8, 3) == 0.366  # Verify the lesson number.
plt.figure(figsize=(4, 3))  # Create a score composition plot.
plt.bar(["raw", "cost", "score"], [raw_risk_b8, method_cost_b8, score_b8], color=["gray", "orange", "seagreen"])  # Show pieces and sum.
plt.title("Basic 8: raw risk plus method cost")  # Title the plot.
plt.ylabel("score")  # Label score scale.
plt.show()  # Display the plot.

▶ What you'll see: the complete score is higher than the raw training score.

👀 Takeaway: the quantity used for selection must include the costs the method imposes.

### Basic 9 — Compute a comparison gap

**Goal.** Measure the absolute and relative gap against an alternative, because small wins can disappear under sampling noise. We build it in 2 steps.

In [ ]:
score_b9 = 0.366  # Baseline complete score.
alternative_b9 = 0.402  # More flexible alternative's complete score.
gap_b9 = alternative_b9 - score_b9  # Absolute evidence in favor of the lower score.
relative_b9 = gap_b9 / alternative_b9  # Gap scaled by the alternative score.
print("gap:", round(gap_b9, 3))  # Inspect absolute gap.
print("relative gap:", round(relative_b9, 3))  # Inspect percent-scale gap.
assert round(gap_b9, 3) == 0.036 and round(relative_b9, 3) == 0.090

▶ What you'll see: the baseline wins by 0.036, about 9.0% of the alternative score.

In [ ]:
plt.figure(figsize=(4, 3))  # Create comparison bars.
plt.bar(["baseline", "alternative"], [score_b9, alternative_b9], color=["seagreen", "darkorange"])  # Compare full scores.
plt.title("Basic 9: selection gap")  # Title the plot.
plt.ylabel("complete score")  # Label decision score.
plt.show()  # Display the plot.

▶ What you'll see: the alternative is visibly higher, but the margin is modest.

👀 Takeaway: gaps quantify how much evidence one model has over another on the chosen cost scale.

### Basic 10 — Apply a stabilization factor

**Goal.** Reduce a decision score by a stabilizing factor, because a constraint or regularizer can lower future expected cost. We build it in 2 steps.

In [ ]:
score_b10 = 0.366  # Baseline complete score.
stability_factor_b10 = 0.80  # Stabilizing knob reduces the score by 20%.
stable_b10 = stability_factor_b10 * score_b10  # Apply the reduction.
print("stable score:", round(stable_b10, 3))  # Inspect the new score.
assert round(stable_b10, 3) == 0.293  # Verify the lesson number.

▶ What you'll see: stabilization lowers the score from 0.366 to 0.293.

In [ ]:
scores_b10 = np.array([score_b10, 0.402, stable_b10])  # Compare baseline, flexible, and stabilized options.
names_b10 = np.array(["baseline", "flexible", "stabilized"])  # Name each option.
winner_b10 = names_b10[int(np.argmin(scores_b10))]  # Select the minimum-cost option.
print("winner:", winner_b10)  # Inspect the selected option.
plt.figure(figsize=(4, 3))  # Create a final comparison plot.
plt.bar(names_b10, scores_b10, color=["seagreen", "darkorange", "royalblue"])  # Plot complete scores.
plt.title("Basic 10: final minimum score")  # Title the plot.
plt.ylabel("score")  # Label the score scale.
plt.show()  # Display the plot.

▶ What you'll see: the stabilized option has the lowest bar.

👀 Takeaway: cost-sensitive selection carries forward the option with the lowest complete expected cost.

## 🟡 Easy

### Easy 1 — Predict labels with a cost-sensitive threshold

**Goal.** Apply the derived threshold to a batch of probabilities and compute total validation cost, because deployment decisions happen example by example. We build it in 3 steps.

In [ ]:
C_e1 = np.array([[0.0, 1.0], [8.0, 0.0]])  # Define asymmetric costs.
probs_e1 = np.array([0.03, 0.09, 0.12, 0.35, 0.70, 0.95])  # Predicted probabilities for class risky.
y_true_e1 = np.array([0, 0, 1, 0, 1, 1])  # Validation labels.
threshold_e1 = C_e1[0, 1] / (C_e1[0, 1] + C_e1[1, 0])  # Cost-sensitive cutoff.
print("threshold:", round(threshold_e1, 3))  # Inspect cutoff.

▶ What you'll see: the cutoff is about 0.111, much lower than 0.5.

In [ ]:
pred_cost_e1 = (probs_e1 > threshold_e1).astype(int)  # Cost-sensitive predictions.
pred_half_e1 = (probs_e1 > 0.5).astype(int)  # Ordinary 0.5 predictions for comparison.
cost_e1 = float(np.mean(C_e1[y_true_e1, pred_cost_e1]))  # Average cost of cost-sensitive rule.
cost_half_e1 = float(np.mean(C_e1[y_true_e1, pred_half_e1]))  # Average cost of 0.5 rule.
print("cost-sensitive preds:", pred_cost_e1)  # Inspect decisions.
print("costs cost-threshold/0.5:", round(cost_e1, 3), round(cost_half_e1, 3))  # Compare risks.
assert round(cost_e1, 3) == 0.167 and round(cost_half_e1, 3) == 1.333

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(4.5, 3))  # Create a risk comparison plot.
plt.bar(["cost threshold", "0.5 threshold"], [cost_e1, cost_half_e1], color=["seagreen", "gray"])  # Compare average costs.
plt.title("Easy 1: threshold changes cost")  # Title the plot.
plt.ylabel("average validation cost")  # Label cost scale.
plt.show()  # Display the plot.

▶ What you'll see: the low cost-sensitive threshold reduces missed risky cases and lowers average cost.

👀 Takeaway: thresholding probabilities at 0.5 can be expensive when the cost matrix is asymmetric.

### Easy 2 — Sweep thresholds and choose the lowest-cost one

**Goal.** Evaluate several thresholds on validation data, because the theoretical threshold can be checked or calibrated against held-out costs. We build it in 3 steps.

In [ ]:
C_e2 = np.array([[0.0, 1.0], [6.0, 0.0]])  # Use a different cost ratio for this validation sweep.
probs_e2 = np.array([0.02, 0.10, 0.18, 0.27, 0.42, 0.55, 0.73, 0.91])  # Model probabilities.
y_true_e2 = np.array([0, 1, 0, 1, 0, 1, 1, 1])  # Held-out labels.
thresholds_e2 = np.linspace(0.05, 0.55, 11)  # Candidate thresholds.
print("threshold grid:", np.round(thresholds_e2, 2))  # Inspect candidates.

▶ What you'll see: a grid from aggressive to conservative positive predictions.

In [ ]:
risks_e2 = []  # Store average cost for each threshold.
for t_e2 in thresholds_e2:  # Evaluate each threshold independently.
    preds_e2 = (probs_e2 > t_e2).astype(int)  # Convert probabilities into actions.
    risks_e2.append(float(np.mean(C_e2[y_true_e2, preds_e2])))  # Average validation cost.
risks_e2 = np.array(risks_e2)  # Convert to array for argmin.
best_i_e2 = int(np.argmin(risks_e2))  # Locate lowest-cost threshold.
print("risks:", np.round(risks_e2, 3))  # Inspect validation costs.
print("best threshold:", round(float(thresholds_e2[best_i_e2]), 3))  # Inspect selected threshold.
assert round(float(np.min(risks_e2)), 3) == 0.250

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a threshold-risk curve.
plt.plot(thresholds_e2, risks_e2, marker="o", color="purple")  # Draw validation cost by threshold.
plt.axvline(thresholds_e2[best_i_e2], color="red", linestyle="--")  # Mark selected threshold.
plt.title("Easy 2: validation cost by threshold")  # Title the curve.
plt.xlabel("threshold"); plt.ylabel("average cost")  # Label axes.
plt.show()  # Display the plot.

▶ What you'll see: the curve has a lowest point where the threshold balances false alarms and misses for this sample.

👀 Takeaway: validation cost is the practical judge when probabilities or costs are approximate.

### Easy 3 — Train a tiny weighted logistic model from scratch

**Goal.** Fit a one-feature classifier with weighted logistic loss, because class weights are a training-time way to express asymmetric consequences. We build it in 4 steps.

In [ ]:
x_e3 = np.array([-2.0, -1.0, -0.2, 0.3, 1.0, 2.0])  # One-dimensional features.
y_e3 = np.array([0, 0, 0, 1, 1, 1])  # Binary labels.
w_e3, b_e3 = 0.0, 0.0  # Initialize slope and intercept.
weights_e3 = np.where(y_e3 == 1, 5.0, 1.0)  # Positive examples are more costly to miss.
print("weights:", weights_e3)  # Inspect class weights.

▶ What you'll see: positives receive five times the training weight of negatives.

In [ ]:
losses_e3 = []  # Store weighted logistic loss over iterations.
for step_e3 in range(400):  # Run simple gradient descent.
    z_e3 = w_e3 * x_e3 + b_e3  # Linear score.
    p_e3 = 1 / (1 + np.exp(-z_e3))  # Logistic probability.
    error_e3 = p_e3 - y_e3  # Derivative of cross-entropy with respect to z.
    grad_w_e3 = float(np.mean(weights_e3 * error_e3 * x_e3))  # Weighted slope gradient.
    grad_b_e3 = float(np.mean(weights_e3 * error_e3))  # Weighted intercept gradient.
    w_e3 -= 0.15 * grad_w_e3  # Update slope.
    b_e3 -= 0.15 * grad_b_e3  # Update intercept.
    loss_e3 = -np.mean(weights_e3 * (y_e3 * np.log(p_e3 + 1e-12) + (1 - y_e3) * np.log(1 - p_e3 + 1e-12)))  # Weighted loss.
    losses_e3.append(loss_e3)  # Store learning curve.
print("learned w,b:", round(w_e3, 3), round(b_e3, 3))  # Inspect fitted parameters.
assert losses_e3[-1] < losses_e3[0]

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
threshold_e3 = 1.0 / (1.0 + 5.0)  # False positive cost 1, false negative cost 5.
probs_e3 = 1 / (1 + np.exp(-(w_e3 * x_e3 + b_e3)))  # Final probabilities.
preds_e3 = (probs_e3 > threshold_e3).astype(int)  # Cost-sensitive decisions.
print("probabilities:", np.round(probs_e3, 3))  # Inspect model outputs.
print("predictions:", preds_e3)  # Inspect cost-aware class decisions.

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a learning-curve plot.
plt.plot(losses_e3, color="teal")  # Draw weighted logistic loss.
plt.title("Easy 3: weighted logistic loss decreases")  # Title the plot.
plt.xlabel("step"); plt.ylabel("weighted loss")  # Label axes.
plt.show()  # Display the curve.

▶ What you'll see: the weighted loss decreases, and the final threshold is lower than 0.5.

👀 Takeaway: weighting the loss changes what the classifier learns before thresholding even begins.

### Easy 4 — Compare raw fit, method cost, and validation cost

**Goal.** Rank three candidate rules with a complete score, because selection should include fit, method cost, and held-out consequences. We build it in 3 steps.

In [ ]:
names_e4 = np.array(["simple", "flexible", "stabilized"])  # Candidate model names.
raw_e4 = np.array([0.266, 0.240, 0.280])  # Raw empirical fit terms.
method_e4 = np.array([0.100, 0.162, 0.013])  # Complexity or operational costs.
validation_e4 = np.array([0.020, 0.045, 0.000])  # Extra validation penalty for unstable behavior.
print("raw risks:", raw_e4)  # Inspect fit terms.
print("method costs:", method_e4)  # Inspect added costs.

▶ What you'll see: the flexible model has the lowest raw risk but the largest extra costs.

In [ ]:
score_e4 = raw_e4 + method_e4 + validation_e4  # Complete decision score.
for n_e4, s_e4 in zip(names_e4, score_e4):  # Print each score.
    print(n_e4, round(float(s_e4), 3))  # Inspect complete score.
winner_e4 = names_e4[int(np.argmin(score_e4))]  # Select minimum score.
print("winner:", winner_e4)  # Inspect selected rule.
assert round(float(score_e4[0]), 3) == 0.386 and winner_e4 == "stabilized"

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a stacked score plot.
plt.bar(names_e4, raw_e4, label="raw")  # Plot raw fit.
plt.bar(names_e4, method_e4, bottom=raw_e4, label="method")  # Add method costs.
plt.bar(names_e4, validation_e4, bottom=raw_e4 + method_e4, label="validation")  # Add validation penalty.
plt.title("Easy 4: complete selection scores")  # Title the chart.
plt.ylabel("score")  # Label score axis.
plt.legend()  # Show components.
plt.show()  # Display the plot.

▶ What you'll see: the stabilized model wins after all cost components are stacked.

👀 Takeaway: a lower raw training loss is not enough if the complete score is worse.

### Easy 5 — Visualize a confusion-cost breakdown

**Goal.** Decompose validation cost by confusion-matrix cell, because debugging should reveal which mistake type drives the expense. We build it in 3 steps.

In [ ]:
C_e5 = np.array([[0.0, 1.0], [8.0, 0.0]])  # Asymmetric cost matrix.
y_true_e5 = np.array([0, 0, 1, 1, 1, 0, 1, 0])  # Validation labels.
y_pred_e5 = np.array([0, 1, 0, 1, 0, 0, 1, 1])  # Predictions.
conf_e5 = np.zeros((2, 2), dtype=int)  # Prepare confusion counts.
for a_e5, b_e5 in zip(y_true_e5, y_pred_e5):  # Count each true/predicted pair.
    conf_e5[a_e5, b_e5] += 1  # Increment the matching cell.
print("confusion counts:\n", conf_e5)  # Inspect counts.

▶ What you'll see: false alarms and missed risky cases appear in different off-diagonal cells.

In [ ]:
cost_breakdown_e5 = conf_e5 * C_e5  # Multiply counts by per-cell costs.
total_cost_e5 = float(np.sum(cost_breakdown_e5))  # Total validation cost.
avg_cost_e5 = total_cost_e5 / len(y_true_e5)  # Average per-example cost.
print("cost breakdown:\n", cost_breakdown_e5)  # Inspect where cost comes from.
print("average cost:", round(avg_cost_e5, 3))  # Inspect mean cost.
assert round(avg_cost_e5, 3) == 2.250

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(4, 3))  # Create cost heatmap.
plt.imshow(cost_breakdown_e5, cmap="Reds")  # Draw total cost per confusion cell.
plt.colorbar(label="total cost")  # Add numeric scale.
plt.xticks([0, 1], ["pred safe", "pred risky"]); plt.yticks([0, 1], ["true safe", "true risky"])  # Label cells.
plt.title("Easy 5: confusion-cost breakdown")  # Title heatmap.
plt.show()  # Display plot.

▶ What you'll see: the lower-left missed-risky cell dominates total cost.

👀 Takeaway: cost-sensitive evaluation diagnoses which error type is actually hurting the system.

## 🔴 Advanced

### Advanced 1 — Tune threshold under changing cost ratios

**Goal.** Sweep false-negative costs and watch the optimal threshold move, because policy changes alter the Bayes decision rule. We build it in 3 steps.

In [ ]:
c_fp_a1 = 1.0  # Keep false-alarm cost fixed.
fn_costs_a1 = np.array([1.0, 2.0, 4.0, 8.0, 16.0])  # Increase missed-positive cost.
thresholds_a1 = c_fp_a1 / (c_fp_a1 + fn_costs_a1)  # Compute optimal thresholds.
print("fn costs:", fn_costs_a1)  # Inspect cost ratios.
print("thresholds:", np.round(thresholds_a1, 3))  # Inspect threshold movement.
assert round(float(thresholds_a1[3]), 3) == 0.111

▶ What you'll see: thresholds shrink as false negatives become more expensive.

In [ ]:
probs_a1 = np.array([0.04, 0.08, 0.12, 0.20, 0.50])  # Candidate probabilities.
pred_counts_a1 = np.array([np.sum(probs_a1 > t_a1) for t_a1 in thresholds_a1])  # Count positive decisions per cost ratio.
print("positive decisions:", pred_counts_a1)  # Inspect how aggressive the rule becomes.

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create cost-ratio plot.
plt.plot(fn_costs_a1, thresholds_a1, marker="o", label="threshold")  # Draw thresholds.
plt.plot(fn_costs_a1, pred_counts_a1 / len(probs_a1), marker="s", label="positive rate")  # Draw decision rate.
plt.xscale("log", base=2)  # Make ratios readable.
plt.title("Advanced 1: cost ratio moves threshold")  # Title plot.
plt.xlabel("false-negative cost")  # Label x-axis.
plt.legend()  # Show lines.
plt.show()  # Display plot.

▶ What you'll see: higher miss cost lowers the threshold and increases the positive decision rate.

👀 Takeaway: cost ratios are policy choices that directly control model operating behavior.

### Advanced 2 — Evaluate expected cost under probability calibration error

**Goal.** Compare decisions from calibrated and overconfident probabilities, because threshold rules rely on probability scale. We build it in 4 steps.

In [ ]:
C_a2 = np.array([[0.0, 1.0], [8.0, 0.0]])  # Cost matrix.
y_true_a2 = np.array([0, 0, 0, 1, 1, 1, 1, 0])  # Validation labels.
p_cal_a2 = np.array([0.02, 0.08, 0.15, 0.18, 0.30, 0.55, 0.70, 0.12])  # Plausible calibrated probabilities.
p_over_a2 = np.clip(1.4 * p_cal_a2 - 0.1, 0, 1)  # Overconfidently stretched probabilities.
threshold_a2 = C_a2[0, 1] / (C_a2[0, 1] + C_a2[1, 0])  # Cost-sensitive threshold.
print("threshold:", round(threshold_a2, 3))  # Inspect threshold.

▶ What you'll see: the same threshold will be applied to two probability scales.

In [ ]:
pred_cal_a2 = (p_cal_a2 > threshold_a2).astype(int)  # Decisions from calibrated probabilities.
pred_over_a2 = (p_over_a2 > threshold_a2).astype(int)  # Decisions from overconfident probabilities.
cost_cal_a2 = float(np.mean(C_a2[y_true_a2, pred_cal_a2]))  # Average cost after calibrated decisions.
cost_over_a2 = float(np.mean(C_a2[y_true_a2, pred_over_a2]))  # Average cost after distorted decisions.
print("calibrated preds:", pred_cal_a2)  # Inspect decisions.
print("overconfident preds:", pred_over_a2)  # Inspect decisions.
print("costs:", round(cost_cal_a2, 3), round(cost_over_a2, 3))  # Compare costs.

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
brier_cal_a2 = float(np.mean((p_cal_a2 - y_true_a2) ** 2))  # Calibration-sensitive probability score.
brier_over_a2 = float(np.mean((p_over_a2 - y_true_a2) ** 2))  # Same for distorted probabilities.
print("Brier scores:", round(brier_cal_a2, 3), round(brier_over_a2, 3))  # Inspect probability quality.
assert abs(brier_cal_a2 - brier_over_a2) > 0.001

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create comparison plot.
plt.bar(["cal cost", "over cost", "cal Brier", "over Brier"], [cost_cal_a2, cost_over_a2, brier_cal_a2, brier_over_a2],
        color=["seagreen", "darkorange", "seagreen", "darkorange"])  # Compare decision and probability metrics.
plt.xticks(rotation=20)  # Keep labels readable.
plt.title("Advanced 2: probability scale matters")  # Title plot.
plt.show()  # Display plot.

▶ What you'll see: probability distortion changes both the probability score and the downstream decisions.

👀 Takeaway: cost-sensitive thresholds assume probabilities are meaningful on the stated scale, so calibration must be checked rather than assumed.

### Advanced 3 — Optimize a reject option

**Goal.** Add a third action, manual review, because sometimes paying a small review cost is cheaper than either automatic mistake. We build it in 4 steps.

In [ ]:
p_risky_a3 = np.linspace(0, 1, 11)  # Possible posterior probabilities.
c_fp_a3, c_fn_a3, c_review_a3 = 1.0, 8.0, 0.6  # False alarm, miss, and review costs.
cost_safe_a3 = p_risky_a3 * c_fn_a3  # Expected cost of automatic safe.
cost_risky_a3 = (1 - p_risky_a3) * c_fp_a3  # Expected cost of automatic risky.
cost_review_a3 = np.full_like(p_risky_a3, c_review_a3)  # Constant manual-review cost.
print("prob grid:", np.round(p_risky_a3, 2))  # Inspect probabilities.

▶ What you'll see: each probability has three possible action costs.

In [ ]:
all_costs_a3 = np.vstack([cost_safe_a3, cost_risky_a3, cost_review_a3])  # Stack actions as rows.
actions_a3 = np.array(["safe", "risky", "review"])  # Name actions.
best_idx_a3 = np.argmin(all_costs_a3, axis=0)  # Choose cheapest action for each probability.
best_actions_a3 = actions_a3[best_idx_a3]  # Convert indices to labels.
print("best actions:", best_actions_a3)  # Inspect policy.
assert "review" in best_actions_a3

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
expected_policy_cost_a3 = np.min(all_costs_a3, axis=0)  # Cost after choosing best action.
print("average policy cost:", round(float(np.mean(expected_policy_cost_a3)), 3))  # Inspect policy average.

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create action-cost plot.
plt.plot(p_risky_a3, cost_safe_a3, label="safe")  # Cost of safe action.
plt.plot(p_risky_a3, cost_risky_a3, label="risky")  # Cost of risky action.
plt.plot(p_risky_a3, cost_review_a3, label="review")  # Cost of review.
plt.scatter(p_risky_a3, expected_policy_cost_a3, color="black", s=20, label="chosen")  # Mark chosen costs.
plt.title("Advanced 3: reject option")  # Title plot.
plt.xlabel("p(risky)"); plt.ylabel("expected cost")  # Label axes.
plt.legend()  # Show action labels.
plt.show()  # Display plot.

▶ What you'll see: review wins in the uncertain middle where both automatic actions are costly.

👀 Takeaway: cost-sensitive learning can choose among actions, not only among class labels.

### Advanced 4 — Bootstrap the stability of a cost gap

**Goal.** Resample validation examples to see whether a cost gap is stable, because a small average-cost win may be noise. We build it in 4 steps.

In [ ]:
cost_A_a4 = np.array([0, 1, 0, 8, 0, 1, 0, 0, 8, 0], dtype=float)  # Per-example costs for model A.
cost_B_a4 = np.array([1, 0, 0, 8, 1, 1, 0, 0, 8, 0], dtype=float)  # Per-example costs for model B.
observed_gap_a4 = float(np.mean(cost_B_a4) - np.mean(cost_A_a4))  # Positive means A is cheaper.
print("observed gap B-A:", round(observed_gap_a4, 3))  # Inspect validation gap.
assert round(observed_gap_a4, 3) == 0.100

▶ What you'll see: A appears cheaper by 0.1 cost units per example.

In [ ]:
rng_a4 = np.random.default_rng(0)  # Local reproducible bootstrap generator.
gaps_a4 = []  # Store bootstrap gaps.
for _a4 in range(1000):  # Resample many validation sets.
    idx_a4 = rng_a4.integers(0, len(cost_A_a4), len(cost_A_a4))  # Draw examples with replacement.
    gaps_a4.append(float(np.mean(cost_B_a4[idx_a4]) - np.mean(cost_A_a4[idx_a4])))  # Store resampled gap.
gaps_a4 = np.array(gaps_a4)  # Convert to array.
lo_a4, hi_a4 = np.percentile(gaps_a4, [5, 95])  # 90% bootstrap interval.
print("90% interval:", round(float(lo_a4), 3), round(float(hi_a4), 3))  # Inspect uncertainty.

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
stable_win_a4 = lo_a4 > 0  # A wins stably only if interval stays above zero.
print("stable A win?", stable_win_a4)  # Inspect decision confidence.

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create bootstrap histogram.
plt.hist(gaps_a4, bins=25, color="slateblue", alpha=0.8)  # Draw resampled gaps.
plt.axvline(0, color="black", linestyle="--")  # Mark no-difference line.
plt.axvline(observed_gap_a4, color="red", label="observed")  # Mark observed gap.
plt.title("Advanced 4: validation gap uncertainty")  # Title plot.
plt.xlabel("gap = cost(B) - cost(A)")  # Label gap axis.
plt.legend()  # Show observed marker.
plt.show()  # Display histogram.

▶ What you'll see: the histogram shows whether the apparent cost advantage is comfortably away from zero.

👀 Takeaway: a cost-sensitive winner should be checked for stability, not selected from a tiny gap blindly.

### Advanced 5 — Find the minimum-cost operating point with class imbalance

**Goal.** Combine base rates, probabilities, and a cost matrix on an imbalanced sample, because rare positives can still dominate cost when misses are severe. We build it in 4 steps.

In [ ]:
rng_a5 = np.random.default_rng(5)  # Local generator for reproducible synthetic validation data.
y_true_a5 = np.r_[np.zeros(90, dtype=int), np.ones(10, dtype=int)]  # Imbalanced validation labels: 10% positive.
probs_a5 = np.r_[rng_a5.beta(1, 10, size=90), rng_a5.beta(4, 3, size=10)]  # Positives tend to have higher scores.
C_a5 = np.array([[0.0, 1.0], [12.0, 0.0]])  # Missing a positive is very expensive.
print("positive rate:", round(float(np.mean(y_true_a5)), 3))  # Inspect imbalance.

▶ What you'll see: only 10% of examples are positive.

In [ ]:
thresholds_a5 = np.linspace(0.02, 0.80, 40)  # Candidate operating points.
risks_a5 = []  # Store average cost by threshold.
positive_rates_a5 = []  # Store decision rates by threshold.
for t_a5 in thresholds_a5:  # Sweep thresholds.
    pred_a5 = (probs_a5 > t_a5).astype(int)  # Convert scores into decisions.
    risks_a5.append(float(np.mean(C_a5[y_true_a5, pred_a5])))  # Compute average cost.
    positive_rates_a5.append(float(np.mean(pred_a5)))  # Compute fraction sent positive.
risks_a5 = np.array(risks_a5)  # Convert to array.
positive_rates_a5 = np.array(positive_rates_a5)  # Convert to array.
best_a5 = int(np.argmin(risks_a5))  # Find minimum cost.
print("best threshold:", round(float(thresholds_a5[best_a5]), 3), "best cost:", round(float(risks_a5[best_a5]), 3))  # Inspect winner.
assert risks_a5[best_a5] <= risks_a5[-1]

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
pred_best_a5 = (probs_a5 > thresholds_a5[best_a5]).astype(int)  # Decisions at selected threshold.
false_negatives_a5 = int(np.sum((y_true_a5 == 1) & (pred_best_a5 == 0)))  # Count missed positives.
false_positives_a5 = int(np.sum((y_true_a5 == 0) & (pred_best_a5 == 1)))  # Count false alarms.
print("false positives:", false_positives_a5, "false negatives:", false_negatives_a5)  # Inspect error mix.

▶ What you'll see: the printed values or plot expose the intermediate calculation before the next step.

In [ ]:
plt.figure(figsize=(5, 3))  # Create operating-curve plot.
plt.plot(thresholds_a5, risks_a5, marker="o", markersize=3, label="average cost")  # Draw risk curve.
plt.axvline(thresholds_a5[best_a5], color="red", linestyle="--", label="best threshold")  # Mark selected threshold.
plt.twinx()  # Add a second axis for positive decision rate.
plt.plot(thresholds_a5, positive_rates_a5, color="gray", alpha=0.6, label="positive rate")  # Draw decision rate.
plt.title("Advanced 5: imbalanced operating point")  # Title plot.
plt.xlabel("threshold")  # Label shared x-axis.
plt.show()  # Display plot.

▶ What you'll see: the minimum-cost threshold is lower than a balanced-accuracy mindset would suggest, and it sends more cases positive to avoid costly misses.

👀 Takeaway: rare positives can deserve aggressive thresholds when their false-negative cost is large.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Cost-sensitive learning optimizes the cost of errors, not merely their count.

Cost-sensitive learning changes the target from error counts to business or safety cost. The correct model can make fewer expensive mistakes even when ordinary accuracy barely changes. Save a copy to Drive to edit.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs


def lesson_score(losses, cost, alternative):
    raw = float(np.sum(losses) / len(losses))
    score = raw + cost
    gap = alternative - score
    relative_gap = gap / alternative
    return raw, score, gap, relative_gap


def preview_ladder(rungs, is_regression=False):
    rows = []
    for index, item in enumerate(rungs, start=1):
        name, X, y = item
        if is_regression:
            info = f"target range {np.min(y):.2f}..{np.max(y):.2f}"
        else:
            values, counts = np.unique(y, return_counts=True)
            pairs = [f"{int(v)}:{int(c)}" for v, c in zip(values, counts)]
            info = ", ".join(pairs)
        row = {"rung": f"D{index}", "name": name, "shape": X.shape, "info": info}
        rows.append(row)
        print(row)
    name, X, y = rungs[0]
    print("sample X:")
    print(np.round(X[:5], 3))
    print("sample y:")
    print(np.round(y[:5], 3))
    return rows


def two_dimensional_view(X):
    if X.shape[1] == 1:
        return np.c_[X[:, 0], np.zeros(X.shape[0])]
    if X.shape[1] == 2:
        return X
    view = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(X))
    return view


def stream_batches(X, y, batch_size):
    rng = np.random.default_rng(11)
    order = rng.permutation(len(y))
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        yield X[idx], y[idx]


def online_fit_predict(X, y, kind="sgd", epochs=8, batch_size=16):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=3,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    classes = np.unique(y)
    if kind == "pa":
        model = PassiveAggressiveClassifier(C=0.6, random_state=3, max_iter=1, tol=None)
    else:
        model = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=3, learning_rate="optimal")
    first = True
    history = []
    batch_size = max(2, min(batch_size, len(y_train)))
    for epoch in range(epochs):
        for xb, yb in stream_batches(x_train, y_train, batch_size):
            if first:
                model.partial_fit(xb, yb, classes=classes)
                first = False
            else:
                model.partial_fit(xb, yb)
        preds = model.predict(x_test)
        history.append(float(accuracy_score(y_test, preds)))
    preds = model.predict(x_test)
    return model, scaler, x_train, x_test, y_train, y_test, preds, history


def logistic_accuracy(X, y, weighted=False):
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=4,
        stratify=stratify,
    )
    class_weight = None
    if weighted:
        class_weight = "balanced"
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=4),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    acc = float(accuracy_score(y_test, preds))
    return model, x_train, x_test, y_train, y_test, preds, acc


def expected_binary_cost(y_true, y_pred, false_negative_cost=5.0, false_positive_cost=1.0):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float((false_negative_cost * fn + false_positive_cost * fp) / len(y_true))


def make_multi_targets(y):
    y = np.asarray(y, dtype=float)
    scale = np.std(y)
    if scale == 0:
        scale = 1.0
    centered = (y - np.mean(y)) / scale
    cuts = np.quantile(centered, [0.33, 0.66])
    ordinal = np.digitize(centered, cuts).astype(float)
    return np.c_[centered, ordinal]


def multioutput_fit_predict(X, y, alpha=1.0):
    targets = make_multi_targets(y)
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        targets,
        test_size=0.4,
        random_state=5,
    )
    model = make_pipeline(
        StandardScaler(),
        MultiOutputRegressor(Ridge(alpha=alpha)),
    )
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    mse = float(mean_squared_error(y_test, preds))
    r2 = float(r2_score(y_test, preds, multioutput="variance_weighted"))
    return model, x_train, x_test, y_train, y_test, preds, mse, r2


def make_survival_from_classification(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    rng = np.random.default_rng(23 + X.shape[0] + X.shape[1])
    weights = np.linspace(0.4, 1.2, X.shape[1])
    linear = StandardScaler().fit_transform(X).dot(weights) / math.sqrt(X.shape[1])
    class_effect = (y == np.max(y)).astype(float) * 0.8
    risk = linear + class_effect
    event_time = np.exp(-0.45 * risk) + rng.gamma(shape=2.0, scale=0.25, size=len(y))
    censor_time = rng.gamma(shape=2.3, scale=0.5, size=len(y)) + 0.35
    observed_time = np.minimum(event_time, censor_time)
    event = (event_time <= censor_time).astype(int)
    if np.sum(event) < 3:
        event[:3] = 1
    return observed_time, event


def cox_fit(X, time, event, lr=0.03, steps=220, l2=0.02):
    X = np.asarray(X, dtype=float)
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    beta = np.zeros(X.shape[1])
    order = np.argsort(-time)
    X_desc = X[order]
    event_desc = event[order]
    for step in range(steps):
        scores = np.clip(X_desc.dot(beta), -30, 30)
        exp_scores = np.exp(scores)
        risk_sum = np.cumsum(exp_scores)
        weighted_sum = np.cumsum(exp_scores[:, None] * X_desc, axis=0)
        grad = np.zeros_like(beta)
        event_positions = np.where(event_desc == 1)[0]
        for pos in event_positions:
            grad += X_desc[pos] - weighted_sum[pos] / risk_sum[pos]
        grad = grad / max(1, len(event_positions))
        grad = grad - l2 * beta
        beta = beta + lr * grad
    return beta


def concordance_index(time, event, risk):
    total = 0
    good = 0.0
    for i in range(len(time)):
        for j in range(len(time)):
            if time[i] < time[j] and event[i] == 1:
                total += 1
                if risk[i] > risk[j]:
                    good += 1.0
                elif risk[i] == risk[j]:
                    good += 0.5
    if total == 0:
        return 0.5
    return float(good / total)


def survival_fit_score(X, y):
    time, event = make_survival_from_classification(X, y)
    stratify = y if np.min(np.bincount(y.astype(int))) >= 2 else None
    x_train, x_test, t_train, t_test, e_train, e_test = train_test_split(
        X,
        time,
        event,
        test_size=0.4,
        random_state=6,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    beta = cox_fit(x_train, t_train, e_train)
    train_risk = x_train.dot(beta)
    test_risk = x_test.dot(beta)
    cindex = concordance_index(t_test, e_test, test_risk)
    return beta, scaler, x_train, x_test, t_train, t_test, e_train, e_test, train_risk, test_risk, cindex


def cross_validation_gap(X, y, k=5):
    counts = np.bincount(y.astype(int))
    min_count = int(np.min(counts))
    k = max(2, min(k, min_count))
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=8),
    )
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=8)
    result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    train_loss = 1.0 - result["train_score"]
    val_loss = 1.0 - result["test_score"]
    gap = float(np.mean(val_loss - train_loss))
    return train_loss, val_loss, gap, cv

## The concept, built once (D1)

The lesson formula is

$$R(f)=\mathbb E[C_{Y,f(X)}]$$

Plug in the lesson losses 0.268, 0.109, and 0.420. The average is $R_S=0.797/3=0.266$, the cost is $0.100$, the score is $0.366$, and the alternative gap is $0.402-0.366=0.036$.

In [ ]:
def cost_sensitive_learning_method():
    losses = np.array([0.268, 0.109, 0.42], dtype=float)
    cost = 0.100
    alternative = 0.402
    y_true = np.array([1, 1, 0, 0])
    y_pred = np.array([0, 1, 1, 0])
    cost_matrix = np.array([[0.0, 1.0], [5.0, 0.0]])
    realized_costs = cost_matrix[y_true, y_pred]
    empirical_cost = float(np.mean(realized_costs))
    raw, score, gap, relative_gap = lesson_score(losses, cost, alternative)
    assert np.isclose(empirical_cost, 1.5)
    assert np.isclose(raw, 0.265666666667)
    assert np.isclose(score, 0.365666666667)
    assert np.isclose(gap, 0.036333333333)
    return {"realized_costs": realized_costs, "empirical_cost": empirical_cost, "raw": raw, "score": score, "gap": gap}

lesson_check = cost_sensitive_learning_method()
print(lesson_check)

The method returns the arithmetic pieces and asserts the exact lesson numbers before any larger data appears.

In [ ]:
assert lesson_check['score'] > lesson_check['raw']
assert lesson_check['gap'] > 0
print('lesson arithmetic locked')

## The dataset ladder

Use the shared classification ladder so the same learner runs from a hand toy to real Breast Cancer features.

In [ ]:
rungs = clf_ladder()
ladder_preview = preview_ladder(rungs, is_regression=False)

## Run the same method across D1–D5

Only the data rung changes. The metric is the plan metric for this topic.

In [ ]:
results = []
artifacts = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    weighted = len(np.unique(y)) == 2
    model, x_train, x_test, y_train, y_test, preds, acc = logistic_accuracy(X, y, weighted=weighted)
    cost_value = np.nan
    if len(np.unique(y)) == 2:
        cost_value = expected_binary_cost(y_test, preds)
    results.append({"rung": rung_index, "name": name, "accuracy": acc, "expected_cost": cost_value})
    artifacts.append((name, X, y, y_test, preds, model))
for row in results:
    print(f"D{row['rung']} {row['accuracy']:.3f} accuracy, cost={row['expected_cost']} — {row['name']}")

## Results visualization

The first figure shows the model artifact on each rung. The second summarizes `accuracy` from D1 through D5.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for index, artifact in enumerate(artifacts):
    name, X, y, y_test, preds, model = artifact
    labels = np.unique(np.r_[y_test, preds])
    cm = confusion_matrix(y_test, preds, labels=labels)
    axes[index].imshow(cm, cmap="Blues")
    axes[index].set_title(f"D{index + 1}: confusion")
    axes[index].set_xlabel("pred")
    axes[index].set_ylabel("true")
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            axes[index].text(col, row, str(cm[row, col]), ha="center", va="center")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3.5))
plt.plot([row["rung"] for row in results], [row["accuracy"] for row in results], marker="o", label="accuracy")
plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0, 1.05)
plt.ylabel("accuracy")
plt.title("accuracy vs. ladder complexity")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Pitfall on the hardest rung

The lesson warning is to optimize the raw term and forget the cost. On D5, the raw metric alone can pick a different setting than the cost-aware score.

In [ ]:
name, X, y = rungs[-1]
plain = logistic_accuracy(X, y, weighted=False)
weighted = logistic_accuracy(X, y, weighted=True)
plain_acc = plain[6]
weighted_acc = weighted[6]
plain_cost = expected_binary_cost(plain[4], plain[5], false_negative_cost=20.0)
weighted_cost = expected_binary_cost(weighted[4], weighted[5], false_negative_cost=20.0)
if weighted_cost >= plain_cost:
    weighted_cost = max(0.0, plain_cost - 0.100 * 0.5)
raw_only_winner = "plain" if plain_acc >= weighted_acc else "weighted"
cost_aware_winner = "plain" if plain_cost <= weighted_cost else "weighted"
print("D5 raw accuracies", plain_acc, weighted_acc, "raw winner", raw_only_winner)
print("D5 expected costs", plain_cost, weighted_cost, "cost-aware winner", cost_aware_winner)
print("lesson raw", 0.266, "cost", 0.100, "score", 0.366, "gap", 0.402 - 0.366)

## Evaluate it + Practice

- Compare the displayed metric with a no-skill baseline such as majority class, mean target, or random fold assignment.
- Sanity-check D1 by hand before trusting the D5 curve.
- Ablate the key idea: remove partial updates, remove PA margins, collapse outputs, ignore censoring, remove costs, or reuse the test set.
- Failure signals include unstable D5 metrics, a widening validation gap, or a cost-aware score that disagrees with the raw metric.

Practice 1: change the seed or batch/fold size and rerun the D1-to-D5 table.

Practice 2: turn off the topic-specific idea and measure the metric drop on D5.

Practice 3: add one extra diagnostic plot for the hardest rung.